In [1]:
import json
import os

import numpy as np

from Pipeline.Model.GallstoneDataSet import GallstoneDataSet
from Pipeline.Model.EvaluationELM import EvaluationELM
from Pipeline.Model.ExtremeLearningMachine import sigmoid

In [2]:
record_dir = '../../Dataset/Record/'
os.makedirs(record_dir, exist_ok=True)

In [3]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()

features_size = gallstone_dataset.x_train_scaled.shape[1]

In [4]:
hidden_size_explore = range(55, 59)
lambda_value_explore = 2.0 ** np.arange(-10, 0)

In [5]:
# 1. Instantiate the Evaluator Object ONCE
evaluator = EvaluationELM(
    x = gallstone_dataset.x_train,
    y = gallstone_dataset.y_train,
    activation_function     = sigmoid,
    elm_initial_state_range = range(41, 45),
    data_split_state_range  = range(1, 4),
    test_size=0.2,
    k_fold=5
)

In [6]:
combination_record , combination_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range   = hidden_size_explore,
    lambda_range        = lambda_value_explore
)

In [7]:
combination_record.to_csv(os.path.join(record_dir, 'combination_record.csv'), index=False)

In [8]:
combination_result_list = EvaluationELM.extract_top_results(combination_result)
best_combination_result = combination_result_list.iloc[0]

In [9]:
dataset_dir = '../../Dataset/JSON/'
config_file = os.path.join(dataset_dir, 'full_model_configs.json')

with open(config_file, 'r') as f:
    saved_configs = json.load(f)

# Extract the value directly; returns None if the config type isn't found
best_hidden_size = next(
    (config["Hidden_Nodes"] for config in saved_configs if config.get("Model_Types") == "Best_Hidden_Nodes"),
    None
)

In [10]:
hidden_size_smaller = range(features_size//3, best_hidden_size)
lambda_value_wider = 1.44 ** np.arange(-20, 10)

optimization_record , optimization_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range=hidden_size_smaller,
    lambda_range=lambda_value_wider
)

In [11]:
optimization_record.to_csv(os.path.join(record_dir, 'optimization_record.csv'), index=False)

In [12]:
optimization_result_list = EvaluationELM.extract_top_results(optimization_result)
best_optimization_result = optimization_result_list.iloc[0]

In [13]:
best_combination_result

Hidden_Nodes                        56
Activation                     sigmoid
Lambda_Value                      0.25
avg_Accuracy_Seed_Mean          0.7369
avg_Accuracy_Seed_Std           0.0752
avg_Accuracy_Seed_Min            0.549
avg_Accuracy_Seed_Max           0.8627
avg_Precision_Seed_Mean         0.7806
avg_Precision_Seed_Std          0.0968
avg_Precision_Seed_Min          0.5556
avg_Precision_Seed_Max             1.0
avg_Recall_Seed_Mean            0.6563
avg_Recall_Seed_Std             0.1141
avg_Recall_Seed_Min                0.4
avg_Recall_Seed_Max               0.88
avg_NPV_Seed_Mean               0.7129
avg_NPV_Seed_Std                0.0739
avg_NPV_Seed_Min                0.5455
avg_NPV_Seed_Max                  0.88
avg_Specificity_Seed_Mean       0.8152
avg_Specificity_Seed_Std        0.0863
avg_Specificity_Seed_Min        0.6538
avg_Specificity_Seed_Max           1.0
avg_F1-Score_Seed_Mean          0.7082
avg_F1-Score_Seed_Std           0.0921
avg_F1-Score_Seed_Min    

In [14]:
best_optimization_result

Hidden_Nodes                        44
Activation                     sigmoid
Lambda_Value                  0.008735
avg_Accuracy_Seed_Mean          0.7232
avg_Accuracy_Seed_Std           0.0629
avg_Accuracy_Seed_Min           0.5686
avg_Accuracy_Seed_Max           0.8627
avg_Precision_Seed_Mean         0.7537
avg_Precision_Seed_Std          0.0848
avg_Precision_Seed_Min            0.56
avg_Precision_Seed_Max             1.0
avg_Recall_Seed_Mean            0.6623
avg_Recall_Seed_Std              0.102
avg_Recall_Seed_Min                0.4
avg_Recall_Seed_Max               0.84
avg_NPV_Seed_Mean               0.7074
avg_NPV_Seed_Std                0.0624
avg_NPV_Seed_Min                0.5714
avg_NPV_Seed_Max                0.8182
avg_Specificity_Seed_Mean       0.7822
avg_Specificity_Seed_Std        0.0879
avg_Specificity_Seed_Min        0.5769
avg_Specificity_Seed_Max           1.0
avg_F1-Score_Seed_Mean          0.7001
avg_F1-Score_Seed_Std           0.0777
avg_F1-Score_Seed_Min    

In [15]:
new_configs_payload = [
    {
        "Model_Types" : "Grid_Combination",
        "Hidden_Nodes": int(best_combination_result['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_combination_result['Lambda_Value'])
    },
    {
        "Model_Types" : "Grid_Optimization",
        "Hidden_Nodes": int(best_optimization_result['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_optimization_result['Lambda_Value'])
    }
]

In [16]:
if os.path.exists(config_file):
    with open(config_file, 'r') as f:
        existing_configs = json.load(f)
else:
    existing_configs = []

existing_configs.extend(new_configs_payload)

with open(config_file, 'w') as f:
    json.dump(existing_configs, f, indent=4)

print(f"Successfully appended to {config_file}. Total configs now: {len(existing_configs)}")

Successfully appended to ../../Dataset/JSON/full_model_configs.json. Total configs now: 7
